# 1. Introduction to OpenSim

## 1.1. Objectives

**Introduction to OpenSim**

[OpenSim](https://opensim.stanford.edu/) is an open-source software that allows users to develop, analyze, and visualize models of the musculoskeletal system, and to generate dynamic simulations of movement [[1]](https://doi.org/10.1109/TBME.2007.901024). OpenSim enables users to create custom studies, including investigating the effects of musculoskeletal geometry, joint kinematics, and muscle-tendon properties on the forces and joint moments that the muscles can produce. With OpenSim, our goal is to provide a framework that allows the biomechanics community to create, share, and extend a library of models and dynamic simulation tools to study and quantify human and animal movement.

**Purpose**

The purpose of this tutorial is to introduce users to the [`opensim`](https://pypi.org/project/opensim/) package, that allows the use of the library in [Jupyter notebooks](https://jupyter.org/) and other Python environments. In this tutorial you will:

* Verify a local OpenSim install for use in Jupyter and other Python environments.
* Create a simple pendulum in OpenSim.
* Visualize the position of the pendulum using [matplotlib](https://matplotlib.org/).

**Format**

This tutorial first verifies your local `opensim` install, then has you create a simple pendulum, simulate it, and visualize its position. As you work through each section, feel free to explore the `opensim` package and modify the code cells on your own.

**Prerequisites**

This notebook runs against a locally installed [`opensim`](https://pypi.org/project/opensim/) package (>= 4.6) plus the other packages in `requirements.txt`. Nothing is downloaded - every input file it needs ships in this folder under `resources/`.

## 1.2. Set up OpenSim

Run the cell below to confirm `opensim` imports; it prints the version and build date. If it raises `ModuleNotFoundError`, install the requirements first (`pip install -r requirements.txt`).

In [ ]:
import opensim as osim
print(osim.GetVersionAndDate())

The cell below switches to an isolated working directory (`_work/Tutorial 1/`, git-ignored) so anything this notebook writes stays out of the repo folder. It also puts the `opensim` package directory on the DLL search path, which Moco's CasADi/Ipopt solver needs on Windows.

In [ ]:
from tutorial_setup import prepare
prepare("Tutorial 1")

## 1.3. Create a pendulum model

In this section we will create a simple pendulum, simulate it using time-stepping integration, and plot the resulting states trajectory.

Now, let's create a pendulum model. We will use `ModelFactory`, a utility class that allows you to construct common OpenSim `Model`s. Here, we will create a single-link pendulum using `ModelFactor::createPendulum()`.

In [ ]:
# Create a single-link pendulum.
pendulum = osim.ModelFactory.createPendulum()

The following image shows the structure of the pendulum. It consists of a body attached to the ground through a pin joint.

215195548-9c713166-5cac-4ac4-9a7a-728ce74ff074.svg

Before simulating a `Model`, you must initialize the system. Use the `initSystem()` method, which both builds the underlying computational system and constructs a `SimTK::State` containing default values for the `Model`'s coordinates (i.e., joint angles and speeds).

In [ ]:
# Initialize the system and obtain the initial state.
state = pendulum.initSystem()

The `Manager` class is the primary tool for creating a time-stepping simulation with an OpenSim `Model`. It manages the integration of the `Model`s system and records a trajectory of states. Run the following cell to construct a `Manager` object, simulate the pendulum model, and export a `TimeSeriesTable` containing the states trajectory.

In [ ]:
# Create a new Manager object for the pendulum model.
manager = osim.Manager(pendulum)

# Initialize the Manager using the state we obtained from Model::initSystem(). This sets
# the initial conditions for the simulation. The pendulum will start at rest positioned
# 90 degrees from the vertical, and therefore will swing down under the influence of 
# gravity during the simulation.
manager.initialize(state)

# Call Manager's "integrate()" method to simulate the pendulum for 10 seconds.
state = manager.integrate(10.0)

# Obtain a table containing the states of the simulation.
statesTable = manager.getStatesTable()

Next, we will extract the times and joint angles from the `TimeSeriesTable` returned by the `Manager`.

In [ ]:
# Extract time values for the x-axis.
times = statesTable.getIndependentColumn()

# Extract the joint angle of the pendulum over time.
angles = statesTable.getDependentColumn('/jointset/j0/q0/value')

# Print number of states
print(f'Number of simulation states in the time interval [0, {state.getTime()}]: {angles.size()}')

The data from the simulation can now be plotted using, for example, `matplotlib`. The following cell plots the joint angle of the pendulum over time.

In [ ]:
import matplotlib.pyplot as plt

# Plot pendulum's joint angle over time.
plt.title("Pendulum Simulation")
plt.plot(times, angles.to_numpy())
plt.xlabel("time (s)")
plt.ylabel("angle (rad)")

## 1.4. Conclusion

In this tutorial you verified a local [OpenSim](https://opensim.stanford.edu/) install by creating and simulating a simple pendulum. Finally, you plotted the position of the pendulum using [matplotlib](https://matplotlib.org/).

## 1.5. Useful Links


> **OpenSim Website:** https://opensim.stanford.edu/
>
> **OpenSim Python Scripting:** https://opensimconfluence.atlassian.net/wiki/spaces/OpenSim/pages/53085346/Scripting+in+Python
>
> **OpenSim API Documentation:** https://simtk.org/api_docs/opensim/api_docs/
> 
> **OpenSim Creator Website:** https://opensimcreator.com/
> 
> **SimTK Website:** https://simtk.org/projects/opensim
> 
> **Biomechanics of Movement Course Videos:** https://www.youtube.com/channel/UCDNGy0KKNLQ-ztcL5h2Z6zA

## 1.6 Acknowledgments

Thanks to [OpenSimColab](https://simtk.org/projects/opencolab) project [[2]](https://doi.org/10.1080/10255842.2022.2104607) for creating the first OpenSim package for Google Colab and notebook-based workflows.

## 1.7. References


> [1] Delp, S. L., Anderson, F. C., Arnold, A. S., Loan, P., Habib, A., John, C. T., Guendelman, E., & Thelen, D. G. (2007). **OpenSim: open-source software to create and analyze dynamic simulations of movement.** *IEEE Transactions on Bio-Medical Engineering*, 54(11), 1940–1950. https://doi.org/10.1109/TBME.2007.901024
>
> [2] Mokhtarzadeh, H., Jiang, F., Zhao, S., & Malekipour, F. (2022). **OpenColab project: OpenSim in Google colaboratory to explore biomechanics on the web.** *Computer Methods in Biomechanics and Biomedical Engineering*, 1–9. https://doi.org/10.1080/10255842.2022.2104607